## Handling the Extract and Transform part

In [1]:
import os
import numpy as np
import pandas as pd
from datetime import date

import chardet
import seaborn as sns
import logging

In [2]:
def detect_encoding(file):
    detector = chardet.universaldetector.UniversalDetector()
    with open(file, "rb") as f:
        for line in f:
            detector.feed(line)
            if detector.done:
                break
    detector.close()
    return detector.result

In [3]:
# loc = "C:\Offline_Docs\Personal\DT_CS_2023\DE_CS_202309\Project\Case_Study_202309_Data"
# loc = r"F:\github\DE_CS_202309\Project\Case_Study_202309_Data"

loc = r"D:\github_windows\DE_CS_202309\Project\Case_Study_202309_Data"
final_files = 'D:\github_windows\DE_CS_202309\Project\Solution'

enc_list = {}

In [4]:
# Detect Character Encoding

file = loc+'\\'+ os.listdir(loc)[0]
encoding = detect_encoding(file)['encoding']
# print("Encoding:", encoding['encoding'])
encoding

'ascii'

In [5]:
count = 0
filenames = []

for file in os.listdir(loc):
  if file.endswith(".csv"):
    filenames.append(file)
    count+=1
    encoding = detect_encoding(loc+'\\'+file)['encoding']
    enc_list[file]=encoding

print(f'Total {count} CSV files present')

Total 130 CSV files present


In [6]:
df_enc_lst= pd.DataFrame.from_dict(enc_list,orient='Index')
df_enc_lst.reset_index(inplace=True)
df_enc_lst.columns=['File_Name','Encoding']
df_enc_lst.head()

,File_Name,Encoding
0,201901_Orders_2019_02_01_03_10_55.csv,ascii
1,201901_Orders_2019_02_04_15_41_32.csv,ascii
2,201902_Orders_2019_03_04_02_26_31.csv,Windows-1252
3,201903_Orders_2019_04_01_20_39_17.csv,Windows-1252
4,201903_Orders_2019_04_04_08_51_54.csv,ISO-8859-1


## Analyze the File names

In [7]:
df_enc_lst["Orders_For"] = df_enc_lst["File_Name"].str.split("_",n=1, expand=True)[0]
df_enc_lst["Date_File_Load"] = df_enc_lst["File_Name"].str.split("_",expand=True,n=2)[2].str.split(".",expand=True)[0]
df_enc_lst['Date_File_Load']= pd.to_datetime(df_enc_lst["Date_File_Load"],format="%Y_%m_%d_%H_%M_%S")
df_enc_lst["Load_Day"]= df_enc_lst["Date_File_Load"].dt.day_name()

In [8]:
df_enc_lst.head()

,File_Name,Encoding,Orders_For,Date_File_Load,Load_Day
0,201901_Orders_2019_02_01_03_10_55.csv,ascii,201901,2019-02-01 03:10:55,Friday
1,201901_Orders_2019_02_04_15_41_32.csv,ascii,201901,2019-02-04 15:41:32,Monday
2,201902_Orders_2019_03_04_02_26_31.csv,Windows-1252,201902,2019-03-04 02:26:31,Monday
3,201903_Orders_2019_04_01_20_39_17.csv,Windows-1252,201903,2019-04-01 20:39:17,Monday
4,201903_Orders_2019_04_04_08_51_54.csv,ISO-8859-1,201903,2019-04-04 08:51:54,Thursday


In [9]:
df_enc_lst.groupby(by='Load_Day').count()

,File_Name,Encoding,Orders_For,Date_File_Load
Load_Day,,,,
Friday,19,19,19,19
Monday,20,20,20,20
Saturday,14,14,14,14
Sunday,21,21,21,21
Thursday,20,20,20,20
Tuesday,18,18,18,18
Wednesday,18,18,18,18


## File Encoding List

In [10]:
df_enc_lst['Encoding'].value_counts()

Encoding
Windows-1252    68
ISO-8859-1      53
ascii            9
Name: count, dtype: int64

## Create consolidated File

In [11]:
df1 = pd.read_csv(loc+'\\'+ os.listdir(loc)[0], delimiter='|', encoding='utf8')
df1.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,7981,CA-2019-103800,04-01-2019,08-01-2019,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",0,0,0.2,5.5512
1,740,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,0,0,0.2,4.2717
2,741,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,0,0,0.2,-64.7748
3,742,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,0,0,0.8,-5.4870
4,1760,CA-2019-141817,06-01-2019,13-01-2019,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,0,0,0.2,4.8840


In [12]:
mega_df = pd.DataFrame(columns=df1.columns)
mega_df

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit


In [13]:
type(np.count_nonzero(filenames))

int

In [14]:
col_cnt = {}

for row, file in enumerate(filenames):
  tmp_df = pd.read_csv(loc+'\\'+ file, delimiter='|',encoding=enc_list[file])
  # pd.concat([mega_df, tmp_df])
  cnt_cols = np.count_nonzero(tmp_df.columns)
  col_cnt[file] = np.count_nonzero(tmp_df.columns)
  if cnt_cols != 21:
    print(f'Total Columns, {np.count_nonzero(tmp_df.columns)}, Filename - {file}, Encoding - {enc_list[file]}')
  else:
    if row != np.count_nonzero(filenames)-1:
      mega_df = pd.concat([mega_df,tmp_df])


Total Columns, 20, Filename - 202206_Orders_2022_07_05_18_29_34.csv, Encoding - Windows-1252


In [15]:
mega_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21446 entries, 0 to 446
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         21446 non-null  object 
 1   Order ID       21446 non-null  object 
 2   Order Date     21446 non-null  object 
 3   Ship Date      21446 non-null  object 
 4   Ship Mode      21446 non-null  object 
 5   Customer ID    21446 non-null  object 
 6   Customer Name  21446 non-null  object 
 7   Segment        21446 non-null  object 
 8   Country        21446 non-null  object 
 9   City           21446 non-null  object 
 10  State          21446 non-null  object 
 11  Postal Code    21446 non-null  object 
 12  Region         21446 non-null  object 
 13  Product ID     21446 non-null  object 
 14  Category       21446 non-null  object 
 15  Sub-Category   21446 non-null  object 
 16  Product Name   21446 non-null  object 
 17  Sales          21446 non-null  object 
 18  Quantity     

In [16]:
mega_df[['Sales','Quantity']] = mega_df[['Sales','Quantity']].astype(float)

In [17]:
mega_df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,7981,CA-2019-103800,04-01-2019,08-01-2019,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",0.0,0.0,0.2,5.5512
1,740,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,0.0,0.0,0.2,4.2717
2,741,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,0.0,0.0,0.2,-64.7748
3,742,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,0.0,0.0,0.8,-5.4870
4,1760,CA-2019-141817,06-01-2019,13-01-2019,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,0.0,0.0,0.2,4.8840


In [18]:
mega_df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')

## Add File with missing column to the main Dataset. 

In [19]:
miss_col = pd.read_csv(loc+"\\"+'202206_Orders_2022_07_05_18_29_34.csv', encoding="Windows-1252", delimiter="|")
miss_col.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount
0,9951,CA-2022-121559,02-06-2022,04-06-2022,Second Class,HW-14935,Helen Wasserman,Corporate,United States,Indianapolis,Indiana,46203,Central,TEC-AC-10001714,Technology,Accessories,Logitech MX Performance Wireless Mouse,39.890,1,0.0
1,9949,CA-2022-121559,02-06-2022,04-06-2022,Second Class,HW-14935,Helen Wasserman,Corporate,United States,Indianapolis,Indiana,46203,Central,OFF-AP-10002945,Office Supplies,Appliances,Honeywell Enviracaire Portable HEPA Air Cleane...,2405.200,8,0.0
2,9948,CA-2022-121559,02-06-2022,04-06-2022,Second Class,HW-14935,Helen Wasserman,Corporate,United States,Indianapolis,Indiana,46203,Central,FUR-CH-10003746,Furniture,Chairs,Hon 4070 Series Pagoda Round Back Stacking Chairs,1925.880,6,0.0
3,9952,CA-2022-121559,02-06-2022,04-06-2022,Second Class,HW-14935,Helen Wasserman,Corporate,United States,Indianapolis,Indiana,46203,Central,OFF-BI-10002072,Office Supplies,Binders,Cardinal Slant-D Ring Binders,17.380,2,0.0
4,4838,CA-2022-106831,02-06-2022,04-06-2022,First Class,FH-14350,Fred Harton,Consumer,United States,Dublin,Ohio,43017,East,OFF-BI-10003429,Office Supplies,Binders,"Cardinal HOLDit! Binder Insert Strips,Extra St...",3.798,2,0.7


In [20]:
miss_col["Profit"] = 0
miss_col.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,9951,CA-2022-121559,02-06-2022,04-06-2022,Second Class,HW-14935,Helen Wasserman,Corporate,United States,Indianapolis,...,46203,Central,TEC-AC-10001714,Technology,Accessories,Logitech MX Performance Wireless Mouse,39.890,1,0.0,0
1,9949,CA-2022-121559,02-06-2022,04-06-2022,Second Class,HW-14935,Helen Wasserman,Corporate,United States,Indianapolis,...,46203,Central,OFF-AP-10002945,Office Supplies,Appliances,Honeywell Enviracaire Portable HEPA Air Cleane...,2405.200,8,0.0,0
2,9948,CA-2022-121559,02-06-2022,04-06-2022,Second Class,HW-14935,Helen Wasserman,Corporate,United States,Indianapolis,...,46203,Central,FUR-CH-10003746,Furniture,Chairs,Hon 4070 Series Pagoda Round Back Stacking Chairs,1925.880,6,0.0,0
3,9952,CA-2022-121559,02-06-2022,04-06-2022,Second Class,HW-14935,Helen Wasserman,Corporate,United States,Indianapolis,...,46203,Central,OFF-BI-10002072,Office Supplies,Binders,Cardinal Slant-D Ring Binders,17.380,2,0.0,0
4,4838,CA-2022-106831,02-06-2022,04-06-2022,First Class,FH-14350,Fred Harton,Consumer,United States,Dublin,...,43017,East,OFF-BI-10003429,Office Supplies,Binders,"Cardinal HOLDit! Binder Insert Strips,Extra St...",3.798,2,0.7,0


In [21]:
mega_df = pd.concat([mega_df,miss_col])

In [22]:
mega_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21680 entries, 0 to 233
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         21680 non-null  object 
 1   Order ID       21680 non-null  object 
 2   Order Date     21680 non-null  object 
 3   Ship Date      21680 non-null  object 
 4   Ship Mode      21680 non-null  object 
 5   Customer ID    21680 non-null  object 
 6   Customer Name  21680 non-null  object 
 7   Segment        21680 non-null  object 
 8   Country        21680 non-null  object 
 9   City           21680 non-null  object 
 10  State          21680 non-null  object 
 11  Postal Code    21680 non-null  object 
 12  Region         21680 non-null  object 
 13  Product ID     21680 non-null  object 
 14  Category       21680 non-null  object 
 15  Sub-Category   21680 non-null  object 
 16  Product Name   21680 non-null  object 
 17  Sales          21680 non-null  float64
 18  Quantity     

## Enriching Data

In [23]:
mega_df["Total_Sales"] = mega_df["Sales"]*mega_df["Quantity"]
mega_df["Price_After_Disc"] = mega_df["Sales"] - (mega_df["Sales"]*mega_df["Discount"])

In [24]:
def calc_unit_cost(df):
  if df["Quantity"] != 0:
    return df["Price_After_Disc"] / df["Quantity"]
  else:
    return 0

In [25]:
def calc_ord_type(df):
  if df["Quantity"] == 0:
    return "Returns"
  else:
    return "Order"

In [26]:
mega_df["Unit_Cost"]=mega_df.apply(lambda x: calc_unit_cost(x), axis=1)
mega_df["Order_Type"] = mega_df.apply(lambda x: calc_ord_type(x), axis=1)

In [27]:
mega_df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Total_Sales,Price_After_Disc,Unit_Cost,Order_Type
0,7981,CA-2019-103800,04-01-2019,08-01-2019,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",0.0,0.0,0.2,5.5512,0.0,0.0,0.0,Returns
1,740,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,Labels,Avery 508,0.0,0.0,0.2,4.2717,0.0,0.0,0.0,Returns
2,741,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,Storage,SAFCO Boltless Steel Shelving,0.0,0.0,0.2,-64.7748,0.0,0.0,0.0,Returns
3,742,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,Binders,GBC Standard Plastic Binding Systems Combs,0.0,0.0,0.8,-5.4870,0.0,0.0,0.0,Returns
4,1760,CA-2019-141817,06-01-2019,13-01-2019,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,0.0,0.0,0.2,4.8840,0.0,0.0,0.0,Returns


In [ ]:
## Exporting to CSV 

# mega_df.to_csv(final_files+"\\"+'Final_File.csv')

# Generating the Master Data

## Customer and Location Table

In [ ]:
mega_df["Postal Code"] = mega_df["Postal Code"].astype('string')

In [ ]:
mega_df['Location_ID']="US-"+mega_df['Region'].str.slice(0,3)+"-"+mega_df['State'].str.slice(0,3)+"-"+mega_df['City'].str.slice(0,3)+"-"+mega_df['Postal Code']

In [ ]:
df_cust = mega_df[['Customer ID','Customer Name','Segment','Location_ID']]
df_cust.info()

In [ ]:
df_cust = df_cust.drop_duplicates()
df_cust.reset_index(inplace=True,drop=True)
df_cust

In [ ]:
df_loc = mega_df[['Country','City','State','Postal Code','Region','Location_ID']]
df_loc.info()

In [ ]:
df_loc = df_loc.drop_duplicates()
df_loc.reset_index(inplace=True,drop=True)
df_loc

In [ ]:
# df_cust.to_csv(final_files+"\\"+'cust_mstr.csv', index=False)
# df_loc.to_csv(final_files+"\\"+'loc_mstr.csv',index=False)

## Product Table

In [28]:
df_prod = mega_df[['Product ID','Category','Sub-Category','Product Name']]
df_prod = df_prod.drop_duplicates()
df_prod.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1890 entries, 0 to 134
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Product ID    1890 non-null   object
 1   Category      1890 non-null   object
 2   Sub-Category  1890 non-null   object
 3   Product Name  1890 non-null   object
dtypes: object(4)
memory usage: 73.8+ KB


In [35]:
lst = [x for x in df_prod[df_prod["Product ID"].duplicated()]["Product ID"]]

In [36]:
lst

['FUR-FU-10004091',
 'FUR-FU-10001473',
 'FUR-BO-10002213',
 'FUR-FU-10004017',
 'FUR-CH-10001146']

In [43]:
df_prod = df_prod[~df_prod["Product ID"].isin(lst)]
df_prod.head()

,Product ID,Category,Sub-Category,Product Name
0,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form..."
1,OFF-LA-10003223,Office Supplies,Labels,Avery 508
2,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving
3,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs
4,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...


In [44]:
# df_prod.to_csv(final_files+"\\"+'prod_mstr.csv', index=False)

## Release Memory

In [ ]:
# del df_cust
# del df_loc
# del mega_df
# del df_prod